# 01 · Clone AgentTTS to Google Drive and prepare HotpotQA

Run this once. It:
1. clones https://github.com/FairyFali/AgentTTS into a folder in your Drive (~600 MB),
2. checks which files the HotpotQA experiment needs that the public repo does **not** include,
3. builds the HotpotQA dev/test splits the repo expects (`tts_test.json`, `tts_test_2_samples500.json`).

The repo builds those splits with an Elasticsearch server (`data/retrieval_QA_data_generator.py`). This notebook does the same thing with an in-memory BM25 index, so it runs on a CPU runtime in under a minute. **No GPU needed for this notebook.**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"   # shared project folder (repo, data, models, runs)
REPO_URL    = "https://github.com/FairyFali/AgentTTS.git"
N_CANDIDATES = 100   # BM25 paragraphs per question, as in the repo/paper. Lower (e.g. 30) = much shorter, faster prompts.

In [4]:
import os, subprocess, shutil
REPO_DIR = f"{PROJECT_DIR}/AgentTTS"
os.makedirs(PROJECT_DIR, exist_ok=True)

if os.path.isdir(f"{REPO_DIR}/.git"):
    print("Repo already in Drive - pulling latest changes")
    print(subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True).stdout)
else:
    # clone to local disk first (git on the Drive mount is slow), then copy into Drive
    tmp = "/content/_AgentTTS_clone"
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, tmp], check=True)
    shutil.copytree(tmp, REPO_DIR, dirs_exist_ok=True, symlinks=True)
    shutil.rmtree(tmp)
    print("Cloned to", REPO_DIR)
!du -sh "{REPO_DIR}"
!ls "{REPO_DIR}" "{REPO_DIR}/code_others" "{REPO_DIR}/data/HotpotQA"

Cloned to /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/AgentTTS
596M	/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/AgentTTS
/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/AgentTTS:
code_chatdev  code_others  data  LICENSE  Readme.md

/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/AgentTTS/code_others:
agent	     archon		 archon_kgqa.py  archon_taskbench.py
AgentTTS.py  archon_hotpotqa.py  archon_main.py  requirements.txt

/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/AgentTTS/data/HotpotQA:
hotpotqa-dev.json  preprocess.py


### What the public repo is missing for HotpotQA
`archon_hotpotqa.py` imports two helper modules and reads data/lookup files that are not in the repository. Notebook 03 does not depend on any of them (it re-implements the pipeline with the same prompts), but it is useful to know.

In [5]:
need = {
    "code_others/utils.py (precision_recall_f1)":            f"{REPO_DIR}/code_others/utils.py",
    "code_others/evaluation_2wiki.py (metric_evaluation)":   f"{REPO_DIR}/code_others/evaluation_2wiki.py",
    "data/HotpotQA/tts_test.json (dev split)":               f"{REPO_DIR}/data/HotpotQA/tts_test.json",
    "data/HotpotQA/tts_test_2_samples500.json (test split)": f"{REPO_DIR}/data/HotpotQA/tts_test_2_samples500.json",
    "code_others/CSV_files/ret_then_gen_hotpotqa2.csv (agent lookup table)": f"{REPO_DIR}/code_others/CSV_files/ret_then_gen_hotpotqa2.csv",
    "data/HotpotQA/hotpotqa-dev.json (raw HotpotQA dev set)": f"{REPO_DIR}/data/HotpotQA/hotpotqa-dev.json",
}
for name, path in need.items():
    print(("OK       " if os.path.exists(path) else "MISSING  ") + name)

MISSING  code_others/utils.py (precision_recall_f1)
MISSING  code_others/evaluation_2wiki.py (metric_evaluation)
MISSING  data/HotpotQA/tts_test.json (dev split)
MISSING  data/HotpotQA/tts_test_2_samples500.json (test split)
MISSING  code_others/CSV_files/ret_then_gen_hotpotqa2.csv (agent lookup table)
OK       data/HotpotQA/hotpotqa-dev.json (raw HotpotQA dev set)


### Build the HotpotQA splits (BM25 instead of Elasticsearch)

In [6]:
%%writefile /content/build_hotpotqa_data.py
"""
Build the HotpotQA retrieval-QA splits used by AgentTTS (archon_hotpotqa.py).

The original repo builds them with data/retrieval_QA_data_generator.py, which needs a
running Elasticsearch server. This script reproduces the same logic with an in-memory
BM25 index (scikit-learn + scipy), so it runs on a plain Colab CPU in a couple of minutes.

Output format (identical keys to the repo):
  {question, answer, ret_answers, ctxs_candidate, ctxs_gt}

Files written (names match what archon_hotpotqa.py loads):
  tts_test.json                   -> dev/search split, examples [0:100]   (repo uses the first 50)
  tts_test_2_samples500.json      -> test split,       examples [500:1000]
"""
import argparse
import json
import os
import random
import re

import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

_TOKEN = re.compile(r"(?u)\b\w\w+\b")


class BM25:
    """Okapi BM25 over a sparse term matrix (k1=1.2, b=0.75, same defaults as Elasticsearch)."""

    def __init__(self, docs, k1=1.2, b=0.75):
        self.vec = CountVectorizer(lowercase=True, token_pattern=_TOKEN.pattern, dtype=np.float32)
        tf = self.vec.fit_transform(docs).tocsc()  # docs x terms
        n_docs = tf.shape[0]
        dl = np.asarray(tf.sum(axis=1)).ravel()
        avgdl = dl.mean()
        df = np.diff(tf.indptr)  # document frequency per term (csc)
        self.idf = np.log(1 + (n_docs - df + 0.5) / (df + 0.5)).astype(np.float32)
        # pre-compute the BM25 weight for every (doc, term) entry
        tf = tf.tocoo()
        denom = tf.data + k1 * (1 - b + b * dl[tf.row] / avgdl)
        w = tf.data * (k1 + 1) / denom * self.idf[tf.col]
        self.W = sparse.csr_matrix((w, (tf.row, tf.col)), shape=tf.shape).T.tocsr()  # terms x docs

    def search(self, query, size):
        q = self.vec.transform([query])
        q.data[:] = 1.0  # each query term counted once (multi_match-like)
        scores = (q @ self.W).toarray().ravel()
        if size >= len(scores):
            return np.argsort(-scores)
        top = np.argpartition(-scores, size)[:size]
        return top[np.argsort(-scores[top])]


def load_corpus(data):
    """Same corpus as data/HotpotQA/preprocess.py: every context paragraph of the dev set (deduplicated)."""
    seen, texts = set(), []
    for item in data:
        for title, sents in item["context"]:
            text = " ".join(sents)
            if text not in seen:
                seen.add(text)
                texts.append(text)
    return texts


def build_split(data, bm25, corpus, start, count, retrieve_number, seed):
    rng = random.Random(seed)
    out = []
    for ex in data[start:start + count]:
        ctx = {t: " ".join(s) for t, s in ex["context"]}
        gt = []
        for title, _ in ex["supporting_facts"]:
            if ctx[title] not in gt:
                gt.append(ctx[title])
        ctxs = gt.copy()
        for i in bm25.search(ex["question"], retrieve_number):
            if corpus[i] not in ctxs:
                ctxs.append(corpus[i])
        rng.shuffle(ctxs)
        out.append({
            "question": ex["question"],
            "answer": ex["answer"],
            "ret_answers": [ctxs.index(c) for c in gt],
            "ctxs_candidate": ctxs,
            "ctxs_gt": gt,
        })
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--hotpot_dev", required=True, help="path to data/HotpotQA/hotpotqa-dev.json from the repo")
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--retrieve_number", type=int, default=100, help="BM25 hits added to the gold paragraphs (repo: 100)")
    ap.add_argument("--dev_count", type=int, default=100)
    ap.add_argument("--test_start", type=int, default=500)
    ap.add_argument("--test_count", type=int, default=500)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    with open(args.hotpot_dev, encoding="utf-8") as f:
        data = json.load(f)
    corpus = load_corpus(data)
    print(f"HotpotQA dev questions: {len(data)} | corpus paragraphs: {len(corpus)}")
    bm25 = BM25(corpus)
    print("BM25 index built.")

    os.makedirs(args.out_dir, exist_ok=True)
    dev = build_split(data, bm25, corpus, 0, args.dev_count, args.retrieve_number, args.seed)
    test = build_split(data, bm25, corpus, args.test_start, args.test_count, args.retrieve_number, args.seed + 1)
    for name, split in [("tts_test.json", dev), (f"tts_test_2_samples{args.test_count}.json", test)]:
        path = os.path.join(args.out_dir, name)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(split, f, indent=1, ensure_ascii=False)
        n_c = np.mean([len(e["ctxs_candidate"]) for e in split])
        chars = np.mean([sum(len(c) for c in e["ctxs_candidate"]) for e in split])
        print(f"wrote {path}: {len(split)} questions, {n_c:.0f} candidates/question, ~{chars/4:.0f} tokens of references/question")


if __name__ == "__main__":
    main()


Writing /content/build_hotpotqa_data.py


In [7]:
DATA_DIR = f"{PROJECT_DIR}/data/hotpotqa"
os.makedirs(f"{PROJECT_DIR}/code", exist_ok=True)
shutil.copy("/content/build_hotpotqa_data.py", f"{PROJECT_DIR}/code/build_hotpotqa_data.py")
!python /content/build_hotpotqa_data.py --hotpot_dev "{REPO_DIR}/data/HotpotQA/hotpotqa-dev.json" --out_dir "{DATA_DIR}" --retrieve_number {N_CANDIDATES}
!ls -lh "{DATA_DIR}"

HotpotQA dev questions: 7405 | corpus paragraphs: 66609
BM25 index built.
wrote /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/data/hotpotqa/tts_test.json: 100 questions, 100 candidates/question, ~13867 tokens of references/question
wrote /content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching/data/hotpotqa/tts_test_2_samples500.json: 500 questions, 100 candidates/question, ~14059 tokens of references/question
total 34M
-rw------- 1 root root  28M Sep 26 08:38 tts_test_2_samples500.json
-rw------- 1 root root 5.6M Sep 26 08:38 tts_test.json


In [8]:
import json
ex = json.load(open(f"{DATA_DIR}/tts_test.json"))[0]
print("Q:", ex["question"]); print("A:", ex["answer"])
print("gold paragraph positions:", ex["ret_answers"], "of", len(ex["ctxs_candidate"]))
for i in ex["ret_answers"]:
    print(f"  [{i+1}]", ex["ctxs_candidate"][i][:160], "...")

Q: Were Scott Derrickson and Ed Wood of the same nationality?
A: yes
gold paragraph positions: [71, 6] of 101
  [72] Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing  ...
  [7] Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director. ...


Done. Next: **02_download_models.ipynb**.